# 第13回 演習：そのグループ分けは、妥当か（解答例・教員用）


---


## 前回からの接続


前回は、銀河の画像を、著者のコード（Affinity Propagation）で約145のグループに分けました。グループの数は、方法まかせでした。その前の第11回は、142か国をk-meansで3つのグループ（最貧・新興・先進）に分けましたが、K＝3はこちらで決めた数でした。2つや5つのほうが、よい分け方ではないのでしょうか。

正解のラベルがないので、「当たったか」では測れません。今日は、ラベルのないグループ分けの良し悪しを測る物差しをいくつか使い、Kをいくつにするかを、根拠をもって決めます。


---


## 今日の分析目標


あなたは国際協力機関の担当者です。前回の3つのグループを報告書に載せる前に、「なぜ3つなのか」を、数の根拠で説明したい。

Kを変えたときの物差しを、リストや辞書に集めて比べ、報告に載せるKを決めます。最後の「目標に答えられたか」で振り返りましょう。


---


## 学習ゴール


この回を終えると、次のことができるようになります。

- 慣性（inertia）がKとともに必ず下がる理由を説明し、エルボーを読める
- シルエット（silhouette）係数の意味を説明し、Kごとに比べられる
- 乱数（random number）の種を変えたときに、分け方がどれだけ揺れるかを確かめられる
- 大陸のような外の区分と、ARIで比べられる
- Kごと・種ごとの結果をリストや辞書に集め、表にして比べられる


---


## 0. 設定


### 最初に実行する：道具を呼び出す設定のセル

- 役割：このノートで使う道具を呼び出し、前回と同じ142か国の表を用意する
- 入力：データの置き場所
- 出力：「セットアップ完了」の表示


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.metrics import adjusted_rand_score

!pip install -q japanize-matplotlib   # 図中の日本語が □ になるのを防ぐ
try:
    import japanize_matplotlib
except Exception:   # japanize が動かなくなったときの保険：同梱フォントを直接登録する
    from importlib.util import find_spec
    from matplotlib import font_manager as fm
    from pathlib import Path
    spec = find_spec('japanize_matplotlib')
    ttf = next(Path(spec.origin).parent.rglob('*.ttf'), None) if spec else None
    if ttf:
        fm.fontManager.addfont(str(ttf))
        plt.rcParams['font.family'] = fm.FontProperties(fname=str(ttf)).get_name()

plt.rcParams['figure.figsize'] = (7, 5)
plt.rcParams['font.size'] = 13
plt.rcParams['axes.unicode_minus'] = False   # マイナス記号の化けを防ぐ

DATA_DIR = 'https://raw.githubusercontent.com/k0heiun0/applied_exercise_preview/main/shared/data'   # データはこのリポジトリから読み込む
df = pd.read_csv(f'{DATA_DIR}/gapminder.csv')
d = df[df['year'] == 2007].copy().reset_index(drop=True)
d['loggdp'] = np.log10(d['gdpPercap'])
X = StandardScaler().fit_transform(d[['lifeExp', 'loggdp']])   # 前回と同じ
print('セットアップ完了')


### 分析の地図：今日はここ

データ解析は「① データの理解と目標の設定 → ② 前処理（preprocessing）とデータ解析 → ③ 結果の解釈と目標との整合」の3つのフェーズを回ります。今日は ③ の目標との整合（その分け方で、報告に耐えるか）が中心です。


In [ ]:
# @title 図：分析の地図
fig, ax = plt.subplots(figsize=(10, 2.6))
ax.axis('off')
phases = ['① データの理解と\n目標の設定', '② 前処理と\nデータ解析', '③ 結果の解釈と\n目標との整合']
colors = ['#0066cc', '#2a9d8f', '#e63946']
for i, (x0, p, c) in enumerate(zip([0.17, 0.5, 0.83], phases, colors), start=1):
    on = i in (3,)
    ax.text(x0, 0.62, p, ha='center', va='center', fontsize=13 if on else 11, color='white',
            bbox=dict(boxstyle='round,pad=0.6', facecolor=c, alpha=0.95 if on else 0.3))
for a0, a1 in [(0.29, 0.365), (0.62, 0.695)]:
    ax.annotate('', xy=(a1, 0.62), xytext=(a0, 0.62),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.show()


---


## 1. 慣性は、Kとともに必ず下がる


前回、分け方の良し悪しを慣性（各国から自分のグループの中心までの距離の2乗の合計）で測りました。Kを2から8まで変えて、慣性を並べてみます。


### おさらい：forとリスト

```python
out = []
for i in range(3):
    out.append(i * 10)   # out は [0, 10, 20]
```

空のリストを作り、`for`の中で`append`で1つずつ足していく書き方でした（第4回）。


### お手本：Kごとの慣性を、リストに集める

- 役割：Kを2〜8に変えてk-meansを行い、慣性をリストに集める
- 入力：標準化（standardization）した`X`
- 出力：7つの慣性


In [ ]:
ks = range(2, 9)
inertias = []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km.inertia_)
print(np.round(inertias, 1))


84.2、53.8、37.1、25.1、19.2、16.0、13.8。一度も上がらずに下がり続けます。

Kを1つ増やせば、いちばん広がったグループを2つに割るだけでも、距離の合計は必ず小さくなります。極端には、Kを国の数（142）にすれば、どの国も自分が中心で、慣性は0です。だから、慣性がいちばん小さいKを選ぶ、とはいきません。


In [ ]:
# @title 図：エルボー（慣性）
plt.plot(list(ks), inertias, 'o-', lw=2.5, color='#0066cc')
plt.xlabel('K（グループの数）'); plt.ylabel('慣性')
plt.show()


かわりに、下がり方がゆるやかになる曲がり角（ひじ、エルボー）を探します。下がり幅は、2→3で30、3→4で17、4→5で12、5→6で6、6→7で3と、だんだん小さくなります。3のあとで小さくなりはじめ、5のあとでもう一度半分になります。はっきりした角はなく、K＝3ともK＝5とも読めます。エルボーは、目で読むぶん、人によって読み方が分かれる物差しです。


---


## 2. シルエット：自分のグループに、しっくり収まっているか


エルボーだけでは、K＝3とも5とも読めて、Kが決まりませんでした。そこで、別の物差しを使います。シルエット係数は、1か国ずつ「今のグループに収まっているか」を −1〜1の数にします。

$$
s_i = \frac{b_i - a_i}{\max(a_i,\ b_i)}
$$

- $a_i$：同じグループのほかの国までの、距離の平均（小さいほど、グループの中でまとまっている）
- $b_i$：いちばん近い別のグループの国までの、距離の平均（大きいほど、ほかのグループから離れている）

$s_i$ が1に近いほど、しっくり収まっています。0に近いと境目、マイナスなら、別のグループのほうが近いということです。全国の平均が`silhouette_score`で、大きいほどよい分け方です。


### お手本：K＝3のシルエット

- 役割：K＝3で分け、全体のシルエットと、国ごとのシルエットを求める
- 入力：`X`
- 出力：全体の値と、マイナスになった国


In [ ]:
km3 = KMeans(n_clusters=3, n_init=10, random_state=42).fit(X)
print(round(silhouette_score(X, km3.labels_), 3))
s = silhouette_samples(X, km3.labels_)         # 国ごとのシルエット
print(d.loc[s < 0, 'country'].tolist())


K＝3のシルエットは0.513。マイナスになったのは、イラク・サントメプリンシペ・トリニダードトバゴの3か国です。トリニダードトバゴとイラクは、前回のTODO④で「境目の国」として見つけた国でした。別の物差しでも、同じ国が境目に出てきます。


### おさらい：辞書に、キーと値をためる

```python
d = {}
for i in range(3):
    d[i] = i * 10        # d は {0: 0, 1: 10, 2: 20}
```

辞書は、名前（キー）と値の組を覚えておく入れ物でした（第3回）。空の辞書`{}`を作り、`for`で1つずつ取り出しながら、`d[キー] = 値`でキーと値をためていきます。


### TODO①：Kごとのシルエットを、辞書に集める

Kを2〜8に変えて、シルエット（全体の値）を辞書`sil`に`{K: 値}`の形で集め、表示してください。どのKで最大になりますか。


In [ ]:
# TODO: K＝2〜8 のシルエットを辞書 sil に集めて表示する

# 解答例①：K ごとのシルエット
sil = {}
for k in ks:
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    sil[k] = silhouette_score(X, labels)
print({k: round(v, 3) for k, v in sil.items()})


K＝2の0.599が最大です。次がK＝5の0.514、その次がK＝3の0.513で、5と3はほぼ同点です。

シルエットはK＝2を推しました。でも、「豊かで長寿の国」と「そうでない国」の2つに分けるだけでは、支援の方針としては大ざっぱすぎます。


### K＝5は、どんな分け方か

エルボーでもシルエットでも、K＝5はK＝3と並ぶ候補でした。K＝5で分けたグループを、図で見てみます（図の数字は、各グループの平均寿命）。


In [ ]:
# @title 図：K＝5 のグループ
km5 = KMeans(n_clusters=5, n_init=10, random_state=42).fit(X)
for j in range(5):
    g = d[km5.labels_ == j]
    plt.scatter(g['gdpPercap'], g['lifeExp'], s=18, alpha=0.8)
    life = g['lifeExp'].mean()
    plt.text(g['gdpPercap'].median(), life, f'{life:.0f}歳',
             fontsize=13)
plt.xscale('log')
plt.xlabel('一人あたり GDP（対数目盛）'); plt.ylabel('平均寿命')
plt.show()


平均寿命が47歳・61歳・73歳・79歳の4段と、もう1つ、GDPの割に寿命が短い9か国（南アフリカ・ボツワナ・イラクなど、平均51歳）のグループに分かれます。K＝3の「最貧」が2段に割れ、そこからGDPの割に短命な国が抜け出した形です。

K＝5は、K＝3より細かい違いを見せてくれます。3と5のどちらを報告に載せるかは、物差しだけでは決まりません。物差しがいちばんよいKが、目的にいちばんよいKとは限りません。目的から決めるのは、5節です。


---


## 3. 乱数の種を変えると、分け方は揺れるか


エルボーとシルエットで、Kの候補は2・3・5に絞れました。前回、k-meansの答えは出発点で変わることを見ました。出発点は乱数で決まるので、乱数の種（`random_state`）を変えると、分け方が変わるかもしれません。報告に載せるなら、分け方が安定していることも確かめたいところです。

分け方がどれだけ同じかは、第11回のARI（2つの分け方の一致度。同じ分け方なら1、でたらめに分けたのと同じくらいなら0）で測ります。


### お手本：出発点1つで、種を5通り

- 役割：`n_init=1`（出発点1つ）で、種を0〜4に変えてK＝3で分け、慣性と、`km3`とのARIを表示する
- 入力：`X`、`km3`
- 出力：5行の結果


In [ ]:
for seed in range(5):
    k1 = KMeans(n_clusters=3, n_init=1, random_state=seed).fit(X)
    ari = adjusted_rand_score(km3.labels_, k1.labels_)
    print(seed, round(k1.inertia_, 2), round(ari, 3))


出発点が1つだと、慣性は54.67〜57.1とばらつき、`km3`とのARIも0.40〜0.73と、分け方が大きく揺れます。どの種でも、`km3`の慣性（53.75）より大きい、つまりまとまりの悪い分け方で止まっています。


### 新しい書き方：条件に合う数を、sumで数える

TODO②では、「ARIが0.95以上の種はいくつか」を数えます。そのための書き方です。


In [ ]:
print(sum(v >= 0.95 for v in [0.4, 0.98, 0.99]))   # 2 と表示される


- `v >= 0.95`は、`v`ごとに`True`か`False`になる（0.4 → False、0.98 → True、0.99 → True）
- `for v in …`は、並びの中身を1つずつ`v`に入れる
- `sum`は`True`を1、`False`を0として足す。だから、条件に合う数（ここでは2）が出る


### TODO②：10通りの種で、揺れを数える

種を0〜9に変えて（`n_init=1`）、`km3`とのARIを辞書`ari_seed`に`{種: ARI}`の形で集めてください。ARIが0.95以上になった種は、10通りのうちいくつですか。


In [ ]:
# TODO: 種0〜9の ARI を辞書 ari_seed に集め、0.95 以上の数を数える

# 解答例②：10通りの種
ari_seed = {}
for seed in range(10):
    k1 = KMeans(n_clusters=3, n_init=1, random_state=seed).fit(X)
    ari_seed[seed] = adjusted_rand_score(km3.labels_, k1.labels_)
print({s: round(v, 2) for s, v in ari_seed.items()})
print(sum(v >= 0.95 for v in ari_seed.values()))


0.95以上は、10通りのうち3通り（種5・8・9、どれも0.98）だけです。出発点1つでは、いつも同じ分け方にたどり着くとは限りません。

だから、前回から`n_init=10`（出発点を10通り試し、慣性のいちばん小さいものを残す）を使ってきました。`n_init=10`なら種を変えても分け方がほとんど揺れないかは、課題で確かめます。


---


## 4. 大陸と比べる


慣性とシルエットは、データの形だけから測る物差し（内部の物差し）でした。もう一つ、グループ分けを外の区分と比べる見方があります。ここでは、グループが大陸とどれくらい重なるかをARIで測ります。

大陸は「正解」ではありません。重なりが大きければ「地理で説明のつく分け方」、小さければ「地理とは別の切り口」と読みます。


### お手本：K＝3と大陸のARI

- 役割：K＝3のグループと大陸のARIを求める
- 入力：`km3`、`d['continent']`
- 出力：ARI


In [ ]:
print(round(adjusted_rand_score(d['continent'], km3.labels_), 3))


0.365。前回のクロス集計で見たとおり、大陸とある程度は重なりますが、同じではありません。重なりが大きいのと小さいのと、どちらがよいかは、目的しだいです。


### TODO③：Kごとに、大陸と比べる

Kを2〜6に変えて、大陸とのARIを辞書`ari_cont`に集め、表示してください。どのKで最大になりますか。


In [ ]:
# TODO: K＝2〜6 の、大陸との ARI を辞書 ari_cont に集めて表示する

# 解答例③：大陸との ARI
ari_cont = {}
for k in range(2, 7):
    labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    ari_cont[k] = adjusted_rand_score(d['continent'], labels)
print({k: round(v, 3) for k, v in ari_cont.items()})


K＝2で0.241、K＝3で0.365（最大）、K＝4で0.226、K＝5で0.205、K＝6で0.202。K＝3のとき、グループがいちばん大陸と重なります。「アフリカ中心の最貧」「ヨーロッパ中心の先進」と、地理の言葉で説明しやすい分け方です。

報告書の読み手は、国を地域の名前で思い浮かべます。地理の言葉で説明できる分け方は、報告で伝わりやすい。今日の目的（報告に載せる）には、これは強みです。


---


## 5. 答えを出す：報告に載せるK


最初の問いに戻ります。ここまでの物差しを、1つの表にまとめて、報告に載せるKを決めます。


### TODO④：Kごとの物差しを、1つの表にする

Kを2〜6に変えて、1行ずつ辞書`{'K': k, '慣性': …, 'シルエット': …, '大陸ARI': …}`をリスト`rows`に集め、`pd.DataFrame(rows)`で表にしてください（第4回・第7回と同じ作り方です）。


In [ ]:
# TODO: K＝2〜6 の慣性・シルエット・大陸 ARI を1つの表にする

# 解答例④：物差しを1つの表に
rows = []
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    rows.append({'K': k, '慣性': km.inertia_,
                 'シルエット': silhouette_score(X, km.labels_),
                 '大陸ARI': adjusted_rand_score(d['continent'], km.labels_)})
print(pd.DataFrame(rows).round(3))


表にすると、3つの物差しが、それぞれ違うKを推していることがわかります。

| 物差し | 推すK | 読み方 |
|---|---|---|
| 慣性（エルボー） | 3か5 | 3のあとと5のあとで、下がり方がゆるやかになる |
| シルエット | 2（5と3がほぼ同点で次） | 2つに分けるのが、形としていちばんきれい |
| 大陸とのARI | 3 | 地理の言葉で説明しやすい |

物差しは1つの正解を出す機械ではなく、判断の材料です。形の物差しだけなら、2も3も5も候補に残ります。決め手は目的です。支援の方針を立てるには、2つでは大ざっぱすぎます。5つは細かい違いを見せますが、報告の読み手には、「最貧・新興・先進」の3段がいちばん伝わりやすい。

答えは「K＝3を報告に載せる」です。

- 形の物差し：3は5とほぼ同点（シルエット0.513と0.514）
- 大陸との重なり：K＝3がいちばん大きい（ARI 0.365）。地理の言葉で説明でき、報告で伝わりやすい
- 補足：K＝5で分かれるGDPの割に短命な9か国と、シルエットがマイナスの3か国（イラクなど）は、別に示す

まだ言えないことも、2つ残っています。

- くっきり分けてよいのか：どの国も、どれか1つのグループに必ず入れている。境目の国を「60%は新興、40%は最貧」のように、確率で分けることはできないか（発展）
- 数でないデータ：ジャンルや年代のような、区分どうしの「近さ」は、どう地図にするか


---


## 目標に答えられたか


- 今日の目標は「3つのグループを報告に載せる前に、なぜ3つなのかを数の根拠で説明する」ことでした。5節の答えで、どの物差しを根拠にしましたか？
- 1節で、慣性がいちばん小さいKを選べないのは、なぜでしたか？
- 2節のシルエットで、1に近い国・マイナスの国は、それぞれどんな国ですか？
- 3節で、出発点1つだと分け方が揺れたのは、なぜですか？


---


## 今日の要点


- 慣性はKを増やすと必ず下がる。いちばん小さいKではなく、下がり方の曲がり角（エルボー）を読む
- シルエットは、「自分のグループへの近さ」と「別のグループからの遠さ」の比べ。1に近いほど、しっくり収まる
- 出発点1つでは、種によって分け方が揺れる。何通りも試して、安定を確かめる
- 外の区分とのARIで、「何で説明のつく分け方か」を読む
- 物差しは判断の材料。形の物差しで候補（2・3・5）を絞り、目的に照らしてKを決め、根拠を示す
- Python：Kごと・種ごとの結果を、リストや辞書に集め、表にして比べる。条件に合う数は`sum(… for …)`で数える


---


## 次回へ


ここまでのデータは、平均寿命やGDPのような数の列でした。

次回は、行も列も区分（映画のジャンル × 公開の年代）の表を扱います。数の列がなくても、「どのジャンルと、どの年代が近いか」を1枚の地図で読めるでしょうか。


---


## 課題（提出）


提出するもの：応用②の答えと、応用③の文章。提出フォームに入力してください。期限はありません。応用①のコードは提出しませんが、②の答えを出すために必要です。


### 応用①（変形）

3節では、出発点1つ（`n_init=1`）だと、種によって分け方が揺れました。今度は、前回から使ってきた`n_init=10`で、種を0〜4の5通りに変えてK＝3で分け、それぞれの慣性と、`km3`とのARIを表示してください。ARIは辞書`ari10`に`{種: ARI}`の形で集めておきましょう。


In [ ]:
ari10 = {}
for seed in range(5):
    k10 = KMeans(n_clusters=3, n_init=10, random_state=seed).fit(X)
    ari10[seed] = adjusted_rand_score(km3.labels_, k10.labels_)
    print(seed, round(k10.inertia_, 2), round(ari10[seed], 3))


<details><summary>詰まったら</summary>

3節のお手本の`n_init=1`を`n_init=10`に変え、ARIを辞書に入れます。
`ari10 = {}` → `for seed in range(5):`の中で`ari10[seed] = adjusted_rand_score(km3.labels_, …)`です。

</details>


### 応用②（判断）

応用①の5つのARIのうち、いちばん小さい値はいくつですか。小数第2位まで答えてください（例: 0.57）。


In [ ]:
print('答え:', round(min(ari10.values()), 2))


<details><summary>詰まったら</summary>

`min(ari10.values())`を`round(…, 2)`します。

</details>


### 応用③（解釈）

3節（`n_init=1`）と応用①（`n_init=10`）の揺れ方を比べて、「報告に載せる3つのグループは、乱数の種しだいで変わってしまうのではないか」と心配する上司に向けて、3行程度で説明してください。


（模範例）

出発点を1つにすると、種によってARIが0.40〜0.73と大きく揺れ、まとまりの悪い分け方で止まることがありました。

出発点を10通り試す設定にすると、種を5通りに変えてもARIの最小値は0.98で、慣性もほぼ同じ（53.75〜53.77）になり、ほとんど同じ分け方に落ち着きます。

報告に載せる3つのグループは、出発点を十分に試して作ったもので、乱数の種しだいで変わることはほとんどありません。


---


## 発展（任意）


### GMMとBIC：国を、確率で分ける

k-meansは、どの国もどれか1つのグループに必ず入れました。混合ガウスモデル（GMM）は、各グループを「中心のまわりに、だ円形に散らばる山」と考え、各国がどの山から来たかを確率で表します。

Kの選び方にも、BICという物差しが使えます。当てはまりのよさに、山の数を増やしたぶんの罰を足したもので、小さいほどよいです。


In [ ]:
from sklearn.mixture import GaussianMixture

bic = {}
for k in range(1, 8):
    g = GaussianMixture(n_components=k, n_init=5,
                        random_state=0).fit(X)
    bic[k] = g.bic(X)
print({k: round(v, 1) for k, v in bic.items()})
print('BIC が最小の K:', min(bic, key=bic.get))


In [ ]:
g3 = GaussianMixture(n_components=3, n_init=5, random_state=0).fit(X)
prob = g3.predict_proba(X).round(2)        # 各国が、各グループから来た確率
pick = ['Japan', 'Iraq', 'Puerto Rico', 'Portugal', 'Mongolia']
for c in pick:
    i = d.index[d['country'] == c][0]
    print(f'{c:12s}', prob[i])


BICはK＝4で最小（547.3）で、K＝3（549.2）との差は1.9とわずかです。エルボー・シルエット・大陸ARIとは、また違うKを推しました。物差しが変われば推すKも変わる、ということです。差がわずかなので、確率は本編の答えと同じK＝3（`g3`）で見ます。

確率を見ると、日本はほぼ1つのグループに確率1で入ります。一方、プエルトリコ（0.32と0.68）、ポルトガル（0.29と0.71）、モンゴル（0.77と0.23）は、2つのグループに確率が分かれます。「どちらとも言える」ことを、数で表せるのがGMMの強みです。

k-meansでシルエットがマイナスだったイラクは、GMMでは確率1で1つのグループに入ります。境目の顔ぶれが違うのは、GMMがグループを、丸ではなくだ円の山とみるからです。

試すなら、`GaussianMixture(n_components=4, …)`で4つの山に分けて、4つ目のグループがどんな国か、要約してみましょう。
